# Batch Fraud Detection Model Training (Spark ML)

## Objective

The objective of this notebook is to train a machine learning model using the preprocessed PaySim transaction dataset.

In this notebook:

- Load prepared training and testing datasets
- Train a Random Forest classification model
- Generate predictions
- Evaluate model performance
- Save the trained Spark ML model

In [1]:
import os

from pyspark.sql import SparkSession

from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator


## Spark Session Setup

Initializes Apache Spark and configures the environment required for model training.

In [2]:
import os

os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.19.10-hotspot"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"


spark = SparkSession.builder \
    .appName("FraudModelTraining") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()


print("Spark Version:", spark.version)

Spark Version: 3.5.9


## Load Training and Testing Dataset

The datasets created during Spark preprocessing are loaded.

Training data is used for model learning, while testing data is used for evaluating model performance.

In [3]:
from pathlib import Path


project_root = Path(
    r"C:\Users\Suji\Desktop\Real-Time-Financial-Fraud-Detection-Pipeline"
)


train_path = project_root / "data" / "spark_train_data"

test_path = project_root / "data" / "spark_test_data"


train_df = spark.read.parquet(str(train_path))

test_df = spark.read.parquet(str(test_path))


print("Training Data:")
train_df.show(5)

print("Testing Data:")
test_df.show(5)

Py4JJavaError: An error occurred while calling o28.parquet.
: java.lang.UnsatisfiedLinkError: 'boolean org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(java.lang.String, int)'
	at org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(Native Method)
	at org.apache.hadoop.io.nativeio.NativeIO$Windows.access(NativeIO.java:793)
	at org.apache.hadoop.fs.FileUtil.canRead(FileUtil.java:1249)
	at org.apache.hadoop.fs.FileUtil.list(FileUtil.java:1454)
	at org.apache.hadoop.fs.RawLocalFileSystem.listStatus(RawLocalFileSystem.java:601)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:1972)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2014)
	at org.apache.hadoop.fs.ChecksumFileSystem.listStatus(ChecksumFileSystem.java:761)
	at org.apache.spark.util.HadoopFSUtils$.listLeafFiles(HadoopFSUtils.scala:180)
	at org.apache.spark.util.HadoopFSUtils$.$anonfun$parallelListLeafFilesInternal$1(HadoopFSUtils.scala:95)
	at scala.collection.TraversableLike.$anonf